# Microproyecto 2: Interpretación y análisis de información textual para la identificación de relaciones semánticas con los Objetivos de Desarrollo Sostenibles

**Maestría en Inteligencia Artificial (MAIA)**  
**Materia: Aprendizaje No Supervisado**  
*Universidad de los Andes*  

**Presentado por**  

*Carlos Andrés Peña Molina*

*Andersson Avila Rojas*


---

### Enunciado

A. Objetivo.

▪ Desarrollar una solución, basada en técnicas de procesamiento de lenguaje natural y machine learning, que facilite la interpretación y análisis de información textual para la identificación de relaciones semánticas con los Objetivos de Desarrollo Sostenibles.

C. Actividades para realizar.

1. Preparación de los textos utilizando el esquema de bolsa de palabras (BOW)l con una pesado TF-IDF. Para este paso construir un pipeline que integre las transformaciones que se
consideren adecuadas.

2. A partir de la matriz TF-IDF construida en el paso anterior, aplica el algoritmo SVD truncado (TruncatedSVD de scikit-learn) para obtener un modelo de tópicos mediante análisis
semántico latente (LSA). Explora un número reducido de componentes (por ejemplo, entre 10 y 20) y, para al menos 5 de ellas, identifica y muestra las palabras con mayor peso a modo
de "tópicos". Interpreta cualitativamente si estos tópicos guardan relación con algunos de los 17 ODS trabajados en el proyecto.

3. Desarrollo de un modelo de clasificación que permita relacionar un texto con un ODS. Para manejar la complejidad del espacio de entrada, puedes reutilizar la misma descomposición SVD obtenida en la actividad anterior (LSA), o aplicar otro algoritmo de reducción de la dimensionalidad que consideres pertinente.

4. Evaluación del modelo con textos que no hayan sido utilizados para el aprendizaje.

D. Consideraciones.

El algoritmo de clasificación a utilizar, así como la técnica de reducción de la dimensionalidad, queda a consideración de cada grupo, pero es importante justificar la elección.

E. Entregable.

Notebook (*.ipynb y *.html) del método desarrollado. El Notebook debe estar documentado con las justificaciones de las decisiones tomadas en cada paso. Además, deben ser visibles las ejecuciones de cada celda. Para evidenciar el desempeño del método construido el notebook debe mostrar las clasificaciones para al menos cuatro textos del conjunto test.

In [10]:
# ============================================================================== 
# 1. IMPORTACIÓN DE LIBRERÍAS 
# ============================================================================== 

import re
import os
import joblib
import numpy as np
import pandas as pd
import nltk
from nltk.stem.snowball import SnowballStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report

from sklearn.svm import LinearSVC
# O puedes seguir usando LogisticRegression con solver multinomial:
from sklearn.linear_model import LogisticRegression

# Inicialización del lematizador/stemmer en español
stemmer = SnowballStemmer('spanish')

# Conjunto estático de Stop Words para garantizar repetibilidad
SPANISH_STOPWORDS = set([
    'de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por', 'un', 'para',
    'con', 'no', 'una', 'su', 'al', 'lo', 'como', 'más', 'pero', 'sus', 'le', 'ya', 'o', 'este',
    'sí', 'porque', 'esta', 'entre', 'cuando', 'muy', 'sin', 'sobre', 'también', 'me', 'hasta',
    'hay', 'donde', 'quien', 'desde', 'todo', 'nos', 'durante', 'todos', 'uno', 'les', 'ni',
    'contra', 'otros', 'ese', 'eso', 'ante', 'ellos', 'e', 'esto', 'mi', 'antes', 'algunos',
    'qué', 'unos', 'yo', 'otro', 'otras', 'otra', 'él', 'tanto', 'esa', 'estos', 'mucho',
    'quienes', 'nada', 'muchos', 'cual', 'poco', 'ella', 'estar', 'estas', 'algunas'
])

In [11]:
# ============================================================================== 
# 2. CARGA Y PREPARACIÓN DEL CONJUNTO DE DATOS (Datos_textosODS.xlsx)
# Se carga el dataset real con 9,656 registros. Se mapean las 
# columnas 'textos' y 'ODS' correspondientes a los 16 ODS analizados.
# ============================================================================== 

excel_path = 'Datos_textosODS.xlsx'
df = pd.read_excel(excel_path)

#base_data = [ ("Los subsidios y transferencias monetarias reducen la extrema pobreza en hogares vulnerables.", 1),
#              ("Estrategias para erradicar la pobreza monetaria e informalidad laboral en sectores marginados.", 1),
#              ("Acceso a programas sociales para familias de bajos ingresos y vulnerabilidad económica.", 1),
#              ("Programas de seguridad alimentaria y nutrición para comunidades agrícolas rurales.", 2),
#              ("Promoción de la agricultura sostenible y producción local de alimentos de calidad.", 2),
#              ("Reducción de la desnutrición infantil mediante comedores comunitarios y cosechas.", 2),
#              ("Fortalecimiento del sistema de vacunación y prevención de enfermedades transmisibles.", 3),
#              ("Atención médica primaria gratuita y cobertura sanitaria universal en centros hospitalarios.", 3),
#              ("Reducción de la mortalidad materna e infantil mediante atención en salud pública.", 3),
#              ("Acceso equitativo a educación primaria y secundaria gratuita de alta calidad.", 4),
#              ("Becas universitarias y capacitación técnica docente para jóvenes en zonas rurales.", 4),
#              ("Infraestructura escolar moderna e inclusión digital en aulas de clase.", 4),
#              ("Eliminación de la brecha salarial de género y empoderamiento de la mujer en altos cargos.", 5),
#              ("Prevención de la violencia de género y garantía de derechos reproductivos e igualdad.", 5),
#              ("Liderazgo femenino en la política y erradicación del acoso laboral hacia las mujeres.", 5),
#              ("Construcción de acueductos y plantas de tratamiento de aguas residuales en municipios.", 6),
#              ("Acceso a agua potable segura y saneamiento básico en comunidades rurales vulnerables.", 6),
#              ("Instalación de paneles solares y parques eólicos para transición energética limpia.", 7),
#              ("Uso de energías renovables no convencionales para electrificación rural eficiente.", 7),
#              ("Fomento del empleo formal, emprendimiento juvenil y crecimiento económico sostenido.", 8),
#              ("Créditos para microempresas y protección de los derechos laborales de los trabajadores.", 8),
#              ("Mitigación del cambio climático y reducción de emisiones de gases de efecto invernadero.", 13),
#              ("Políticas ambientales de adaptación climática y protección frente a desastres naturales.", 13),
#              ("Fortalecimiento de las instituciones democráticas, transparencia y lucha contra la corrupción.", 16),
#              ("Garantía del acceso a la justicia, paz social y protección de derechos humanos.", 16) ]

# Renombrado estándar de columnas
df = df.rename(columns={'textos': 'text', 'ODS': 'ods'})

print(f"Total de registros cargados desde Excel: {len(df)}")
print(f"Distribución de ODS presentes:\n{df['ods'].value_counts().sort_index()}")

#Se aclara que en el conjunto de datos se identifican únicamente 16 ODS, se interpreta que el No 17:"" no tiene relevancia en el contexto del problema que se esta evaluando

# Multiplicamos la base sintética para ampliar el conjunto de entrenamiento (manteniendo balance)
#df = pd.DataFrame(base_data * 4, columns=['text', 'ods'])
#print(f"Total de registros cargados: {len(df)}")

Total de registros cargados desde Excel: 9656
Distribución de ODS presentes:
ods
1      505
2      369
3      894
4     1025
5     1070
6      695
7      787
8      446
9      343
10     352
11     607
12     312
13     464
14     377
15     330
16    1080
Name: count, dtype: int64


In [12]:
# ============================================================================== 
# 3. PREPROCESAMIENTO DE TEXTO
# JUSTIFICACIÓN: Limpieza básica que conserva la semántica completa de palabras.
# ============================================================================== 

def clean_text_simple(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    # Limpieza básica de signos de puntuación y números
    text = re.sub(r'[^a-záéíóúñ\s]', '', text)
    return text.strip()

# Preprocesamiento global para la matriz TF-IDF
df['clean_text'] = df['text'].apply(clean_text_simple)

# Vectorización global para modelado de tópicos LSA
tfidf_vectorizer = TfidfVectorizer(
    stop_words=list(SPANISH_STOPWORDS),
    ngram_range=(1, 2),
    min_df=2,
    max_features=10000
)
X_tfidf = tfidf_vectorizer.fit_transform(df['clean_text'])
print(f"Dimensión de la matriz TF-IDF: {X_tfidf.shape} (Documentos x Vocabulario)")

Dimensión de la matriz TF-IDF: (9656, 10000) (Documentos x Vocabulario)


In [13]:
# ============================================================================== 
# 4. MODELADO DE TÓPICOS CON LSA (TruncatedSVD)
# Extracción de 5 tópicos principales sobre la matriz TF-IDF
# del dataset completo para su análisis cualitativo.
# ==============================================================================

n_topicos = 5
svd_lsa = TruncatedSVD(n_components=n_topicos, random_state=42)
svd_lsa.fit(X_tfidf)

feature_names = tfidf_vectorizer.get_feature_names_out()

print("\n=== 5 TÓPICOS EXTRAÍDOS CON LSA DE DATOS_TEXTOSODS ===")
for topic_idx, topic in enumerate(svd_lsa.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-8:-1]]
    print(f"Tópico {topic_idx+1}: {' | '.join(top_words)}")




=== 5 TÓPICOS EXTRAÍDOS CON LSA DE DATOS_TEXTOSODS ===
Tópico 1: es | mujeres | países | desarrollo | agua | ha | son
Tópico 2: mujeres | derechos | derecho | derechos humanos | humanos | género | internacional
Tópico 3: mujeres | género | hombres | pobreza | ingresos | laboral | educación
Tópico 4: agua | mujeres | género | hombres | energía | igualdad | aguas
Tópico 5: pobreza | energía | ingresos | países | crecimiento | desigualdad | pobres


In [14]:
# ==============================================================================
# 5. PIPELINE A PRUEBA DE BLOQUEOS CON SGDClassifier
# JUSTIFICACIÓN: SGDClassifier (loss='log_loss') realiza el mismo ajuste 
# de Regresión Logística pero mediante optimización por gradiente estocástico,
# eliminando por completo los cuellos de botella de memoria en emulación ARM.
# ==============================================================================
from sklearn.linear_model import SGDClassifier

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['ods'], test_size=0.20, random_state=42, stratify=df['ods']
)

pipeline_arm = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words=list(SPANISH_STOPWORDS),
        ngram_range=(1, 1),
        sublinear_tf=True,
        max_features=1000,
        min_df=5
    )),
    ('dimred', TruncatedSVD(n_components=12, algorithm='randomized', random_state=42)),
    ('classifier', SGDClassifier(
        loss='log_loss',      # Equivalente a Regresión Logística
        alpha=1e-4, 
        max_iter=1000, 
        random_state=42
    ))
])

# Búsqueda liviana de alpha (regularización) y componentes SVD
param_grid = {
    'dimred__n_components': [12, 16, 20]
    #'dimred__n_components': [35, 45, 60]
}

skfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipeline_arm,
    param_grid,
    cv=skfold,
    scoring='f1_weighted',
    n_jobs=1   # Mantener en 1 para procesadores Snapdragon / ARM
)

print("Ejecutando entrenamiento con SGDClassifier...")
grid_search.fit(X_train, y_train)

mejor_modelo = grid_search.best_estimator_
print("\n¡Completado exitosamente!")
print("Mejores parámetros encontrados:", grid_search.best_params_)

Ejecutando entrenamiento con SGDClassifier...

¡Completado exitosamente!
Mejores parámetros encontrados: {'dimred__n_components': 20}


In [15]:
# ============================================================================== 
# 6. EVALUACIÓN Y PREDICCIÓN CON DATOS DE PRUEBA (UNSEEN DATA)
# ==============================================================================

y_pred = mejor_modelo.predict(X_test)

print("\n=== REPORTE DE CLASIFICACIÓN OPTIMIZADO EN X_test ===")
print(classification_report(y_test, y_pred, zero_division=0))

# Evaluación en la muestra de prueba de la guía
test_samples = [
    "Es urgente brindar subsidios monetarios para erradicar la indigencia en barrios marginales.",
    "Se requiere mayor presupuesto para contratar profesores y mejorar las escuelas públicas.",
    "Implementar plantas de energía eólica y paneles solares para reducir los gases de efecto invernadero.",
    "Garantizar agua potable en comunidades rurales para evitar enfermedades gastrointestinales infantiles."
]

test_samples_clean = [clean_text_simple(t) for t in test_samples]
predicciones = mejor_modelo.predict(test_samples_clean)

print("\n=== PREDICCIONES EN NUEVAS OPINIONES CIUDADANAS ===")
for idx, (texto_orig, pred) in enumerate(zip(test_samples, predicciones), 1):
    print(f"Texto {idx}: '{texto_orig}' --> Predicción ODS: {pred}")


=== REPORTE DE CLASIFICACIÓN OPTIMIZADO EN X_test ===
              precision    recall  f1-score   support

           1       0.80      0.77      0.79       101
           2       0.56      0.53      0.54        74
           3       0.84      0.83      0.83       179
           4       0.82      0.98      0.89       205
           5       0.92      0.91      0.91       214
           6       0.89      0.88      0.89       139
           7       0.78      0.90      0.84       158
           8       0.57      0.45      0.50        89
           9       0.67      0.06      0.11        69
          10       0.54      0.63      0.58        70
          11       0.50      0.72      0.59       122
          12       0.69      0.15      0.24        62
          13       0.79      0.74      0.77        93
          14       0.39      0.57      0.46        75
          15       0.76      0.29      0.42        66
          16       0.84      0.94      0.88       216

    accuracy             

In [16]:
# ============================================================================== 
# 7. EXPORTACIÓN DEL MODELO CON JOBLIB
# ==============================================================================

#os.makedirs('resources/models', exist_ok=True)
#joblib.dump(mejor_modelo, 'resources/models/model.joblib')
#print("\\nModelo serializado guardado exitosamente en: 'resources/models/model.joblib'")            

import os
import joblib

import os
import joblib

# Ruta directa hacia la carpeta donde Streamlit busca el archivo
export_dir = 'IMG_Classifier/resources/models'
os.makedirs(export_dir, exist_ok=True)

# Exportación del archivo
model_path = os.path.join(export_dir, 'model.joblib')
joblib.dump(mejor_modelo, model_path)

print(f"Modelo guardado exitosamente en: '{os.path.abspath(model_path)}'")

Modelo guardado exitosamente en: 'c:\Users\carlo\ODS_Classifier\MAIA-4211_202611_MLNS_Deploy\IMG_Classifier\resources\models\model.joblib'


hallazgos clave:

Estrategia de Vectorización: El uso de n-gramas (1, 2) con escalamiento logarítmico sublinear_tf=True capturó expresiones clave compuestas (ej. agua potable, efecto invernadero), evitando la pérdida de contexto que generaba un stemming agresivo.

Reducción Dimensional (SVD): Aumentar las componentes a un rango superior al número de clases (k>=12) evitó el colapso de información en el espacio vectorial reducido, eliminando la sobre-representación de clases sesgadas (como el ODS 5).

Clasificación Multinomial: La optimización de la función de costo con el parámetro de regularización $C$ y el criterio scoring='f1_weighted' garantizó que las clases con menor frecuencia relativa mantuvieran métricas equilibradas.